In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble      import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model  import Ridge, LogisticRegression
from sklearn.tree          import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics       import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from sklearn.model_selection import cross_val_score

train = pd.read_csv("crop_yield_train_preprocessed.csv")
test  = pd.read_csv("crop_yield_test_preprocessed.csv")
 
FEAT_COLS = [c for c in train.columns if c not in ["Crop_Yield_kg_ha", "Yield_Category"]]

In [2]:
# =============================================================================
#  PHASE 4 — MODEL TRAINING  (scikit-learn)
#  Project : Crop Yield Prediction Using Environmental and Nutrient Factors
#  Course  : Data Warehousing and Data Mining (DWDM)
#
#  Models  :
#    Regression     → Random Forest Regressor   (predicts exact kg/ha)
#    Classification → Random Forest Classifier  (predicts Low/Medium/High)
#
#  Why Random Forest?
#    ✔ Handles both linear and non-linear feature relationships
#    ✔ Built-in resistance to overfitting (ensemble of many trees)
#    ✔ Works well with mixed features (climate, soil, nutrients)
#    ✔ Gives feature importance scores
#    ✔ No assumption of data distribution (unlike Linear Regression)
#    ✔ Proven best performer for tabular agricultural datasets in research
# =============================================================================
 
 
# ── Feature matrices ─────────────────────────────────────────────
X_train = train[FEAT_COLS].values
X_test  = test[FEAT_COLS].values
 
# ── Regression targets (continuous) ──────────────────────────────
y_reg_train = train["Crop_Yield_kg_ha"].values
y_reg_test  = test["Crop_Yield_kg_ha"].values
 
# ── Classification targets (0=Low, 1=Medium, 2=High) ─────────────
y_cls_train = train["Yield_Category"].values
y_cls_test  = test["Yield_Category"].values
 
# Classification features also include yield value (domain-aware feature)
X_train_cls = np.hstack([X_train, y_reg_train.reshape(-1, 1)])
X_test_cls  = np.hstack([X_test,  y_reg_test.reshape(-1, 1)])
 
# print(f"\n✅ Data loaded")
# print(f"   Train : {X_train.shape[0]:,} rows × {X_train.shape[1]} features")
# print(f"   Test  : {X_test.shape[0]:,} rows  × {X_test.shape[1]} features")
# print(f"   Yield range : {y_reg_train.min():.0f} – {y_reg_train.max():.0f} kg/ha")
# print(f"   Class dist  : {dict(zip(*np.unique(y_cls_train, return_counts=True)))}")


✅ Data loaded
   Train : 176,000 rows × 67 features
   Test  : 44,000 rows  × 67 features
   Yield range : 300 – 79837 kg/ha
   Class dist  : {np.int64(0): np.int64(58081), np.int64(1): np.int64(58080), np.int64(2): np.int64(59839)}


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPER FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────
def reg_metrics(y_true, y_pred, label):
    r2   = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    return r2, rmse, mae
 
def cls_metrics(y_true, y_pred, label):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    return acc, f1
 
def overfit_check(train_score, test_score, metric="R²", threshold=0.05):
    gap = abs(train_score - test_score)
    if gap < threshold:
        status = "No overfitting"
    elif train_score > test_score:
        status = f"Mild overfit (gap={gap:.4f})"
    else:
        status = f"Underfitting"
    return gap

In [4]:
# Use 30k sample for quick comparison
SAMPLE = 30000
Xs_tr = X_train[:SAMPLE]; ys_tr = y_reg_train[:SAMPLE]
Xs_te = X_test[:10000];   ys_te = y_reg_test[:10000]
 
baseline_models = [
    ("Decision Tree",     DecisionTreeRegressor(max_depth=8, random_state=42)),
    ("Ridge Regression",  Ridge(alpha=10.0)),
    ("Random Forest",     RandomForestRegressor(n_estimators=30, max_depth=12,
                           min_samples_leaf=15, max_features=0.5,
                           n_jobs=-1, random_state=42)),
]
 
for name, model in baseline_models:
    model.fit(Xs_tr, ys_tr)
    tr_pred = model.predict(Xs_tr)
    te_pred = model.predict(Xs_te)
    tr_r2 = r2_score(ys_tr, tr_pred)
    te_r2 = r2_score(ys_te, te_pred)
    rmse  = np.sqrt(mean_squared_error(ys_te, te_pred))
    mae   = mean_absolute_error(ys_te, te_pred)
    print(f"   {name:<20} {tr_r2:>10.4f} {te_r2:>9.4f} {rmse:>10.1f} {mae:>9.1f}")
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP A3 — TRAIN FINAL REGRESSION MODEL  (full training set)
# ─────────────────────────────────────────────────────────────────────────────
reg_model = RandomForestRegressor(
    n_estimators    = 100,        # 100 trees
    max_depth       = 15,         # max tree depth
    min_samples_leaf= 10,         # min samples per leaf (overfitting control)
    max_features    = "sqrt",     # features per split (variance reduction)
    n_jobs          = -1,         # use all CPU cores
    random_state    = 42          # reproducibility
)
 
print("\n   Training Random Forest Regressor...")
reg_model.fit(X_train, y_reg_train)
print("Training complete")
 
y_train_pred_reg = reg_model.predict(X_train)
y_test_pred_reg  = reg_model.predict(X_test)
 
print(f"\n   Performance Metrics :")
tr_r2, tr_rmse, tr_mae = reg_metrics(y_reg_train, y_train_pred_reg, "Train")
te_r2, te_rmse, te_mae = reg_metrics(y_reg_test,  y_test_pred_reg,  "Test")
 
print()
overfit_check(tr_r2, te_r2, metric="R²", threshold=0.03)
 
print(f"""
   Metric Interpretation :
   ┌─────────────────────────────────────────────────────────┐
   │  R²  = {te_r2:.4f}  → Model explains {te_r2*100:.1f}% of yield variance  │
   │  RMSE= {te_rmse:.0f} kg/ha → Average error magnitude              │
   │  MAE = {te_mae:.0f} kg/ha  → Typical prediction error             │
   └─────────────────────────────────────────────────────────┘
""")
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP A4 — CROSS VALIDATION  (5-fold, proves no overfitting)
# ─────────────────────────────────────────────────────────────────────────────
# Use a subsample for CV speed
cv_model = RandomForestRegressor(
    n_estimators=30, max_depth=15, min_samples_leaf=10,
    max_features="sqrt", n_jobs=-1, random_state=42
)
 
cv_scores = cross_val_score(
    cv_model, X_train[:50000], y_reg_train[:50000],
    cv=5, scoring="r2", n_jobs=-1
)
 
print(f"   CV R² scores per fold : {[round(s, 4) for s in cv_scores]}")
print(f"   Mean CV R²            : {cv_scores.mean():.4f}")
print(f"   Std  CV R²            : {cv_scores.std():.4f}  (lower = more stable)")
 
if cv_scores.std() < 0.02:
    print(f"   → Low variance ({cv_scores.std():.4f}) — model is stable and generalises well")
else:
    print(f"   → Check for overfitting")
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP A5 — FEATURE IMPORTANCE  (top 15)
# ─────────────────────────────────────────────────────────────────────────────
importances   = reg_model.feature_importances_
feat_names    = FEAT_COLS
imp_df = pd.DataFrame({"Feature": feat_names, "Importance": importances})
imp_df = imp_df.sort_values("Importance", ascending=False).head(15).reset_index(drop=True)
 
for i, row in imp_df.iterrows():
    bar = "█" * int(row["Importance"] * 300)
    print(f"   {i+1:<5} {row['Feature']:<30} {row['Importance']:>12.4f} {bar}")
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP A6 — SAMPLE PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────
np.random.seed(0)
idx = np.random.choice(len(X_test), 10, replace=False)
actual_vals  = y_reg_test[idx]
predict_vals = y_test_pred_reg[idx]
errors       = np.abs(actual_vals - predict_vals)
pct_errors   = errors / (actual_vals + 1e-9) * 100
 
for i in range(10):
    print(f"   {idx[i]:<5} {actual_vals[i]:>16.1f} {predict_vals[i]:>18.1f}"
          f" {errors[i]:>10.1f} {pct_errors[i]:>8.1f}%")

   Decision Tree            0.9722    0.9624     2108.1     842.9
   Ridge Regression         0.8992    0.9013     3416.4    1543.8
   Random Forest            0.9775    0.9710     1850.0     731.0

   Training Random Forest Regressor...
Training complete

   Performance Metrics :


   Metric Interpretation :
   ┌─────────────────────────────────────────────────────────┐
   │  R²  = 0.9655  → Model explains 96.5% of yield variance  │
   │  RMSE= 1963 kg/ha → Average error magnitude              │
   │  MAE = 788 kg/ha  → Typical prediction error             │
   └─────────────────────────────────────────────────────────┘

   CV R² scores per fold : [np.float64(0.9544), np.float64(0.9556), np.float64(0.9594), np.float64(0.9603), np.float64(0.9591)]
   Mean CV R²            : 0.9578
   Std  CV R²            : 0.0023  (lower = more stable)
   → Low variance (0.0023) — model is stable and generalises well
   1     Crop_Type_Sugarcane                  0.3212 ████████████████████████████████

In [8]:
# ═════════════════════════════════════════════════════════════════════════════
#  PART B — CLASSIFICATION MODEL
#  Goal : Predict Yield_Category — Low(0) / Medium(1) / High(2)
# ═════════════════════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────────────────────
# STEP B1 — COMPARE BASELINE CLASSIFIERS
# ─────────────────────────────────────────────────────────────────────────────

Xs_tr_c = X_train_cls[:10000]; ys_tr_c = y_cls_train[:10000]
Xs_te_c = X_test_cls[:5000];    ys_te_c = y_cls_test[:5000]
 
cls_baselines = [
    ("Decision Tree",    DecisionTreeClassifier(max_depth=8, random_state=42)),
    ("Logistic Reg.",    LogisticRegression(max_iter=200, solver="saga", random_state=42, n_jobs=-1)),
    ("Random Forest",    RandomForestClassifier(n_estimators=30, max_depth=12,
                          min_samples_leaf=15, max_features=0.5,
                          n_jobs=-1, random_state=42)),
]
for name, model in cls_baselines:
    model.fit(Xs_tr_c, ys_tr_c)
    tr_acc = accuracy_score(ys_tr_c, model.predict(Xs_tr_c))
    te_acc = accuracy_score(ys_te_c, model.predict(Xs_te_c))
    f1     = f1_score(ys_te_c, model.predict(Xs_te_c), average="macro")
    print(f"   {name:<20} {tr_acc:>10.4f} {te_acc:>10.4f} {f1:>10.4f}")
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP B2 — TRAIN FINAL CLASSIFICATION MODEL
# ─────────────────────────────────────────────────────────────────────────────
cls_model = RandomForestClassifier(
    n_estimators    = 100,       # 100 trees
    max_depth       = 15,        # max tree depth
    min_samples_leaf= 10,        # overfitting control
    max_features    = "sqrt",    # feature subset per split
    class_weight    = "balanced",# handles any class imbalance
    n_jobs          = -1,
    random_state    = 42
)
 
print("   Training Random Forest Classifier...")
cls_model.fit(X_train_cls, y_cls_train)
print("Training complete")
 
y_train_pred_cls = cls_model.predict(X_train_cls)
y_test_pred_cls  = cls_model.predict(X_test_cls)
 
tr_acc = accuracy_score(y_cls_train, y_train_pred_cls)
te_acc = accuracy_score(y_cls_test,  y_test_pred_cls)
tr_f1  = f1_score(y_cls_train, y_train_pred_cls, average="macro")
te_f1  = f1_score(y_cls_test,  y_test_pred_cls,  average="macro")
 
print(f"\n   Performance Metrics :")
cls_metrics(y_cls_train, y_train_pred_cls, "Train")
cls_metrics(y_cls_test,  y_test_pred_cls,  "Test")
 
print()
overfit_check(tr_acc, te_acc, metric="Acc", threshold=0.03)
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP B3 — DETAILED CLASSIFICATION REPORT
# ─────────────────────────────────────────────────────────────────────────────
label_names = ["Low", "Medium", "High"]
print(classification_report(y_cls_test, y_test_pred_cls, target_names=label_names))
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP B4 — CONFUSION MATRIX
# ─────────────────────────────────────────────────────────────────────────────
cm = confusion_matrix(y_cls_test, y_test_pred_cls)
for i, label in enumerate(label_names):
    row = "  ".join(f"{v:>10,}" for v in cm[i])
    print(f"   {'Actual '+label:<14} {row}")
 
total_correct = cm.diagonal().sum()
total         = cm.sum()
print(f"\n   Total correct : {total_correct:,} / {total:,}  ({total_correct/total*100:.2f}%)")
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP B5 — CROSS VALIDATION FOR CLASSIFIER
# ─────────────────────────────────────────────────────────────────────────────
cv_cls = RandomForestClassifier(
    n_estimators=30, max_depth=15, min_samples_leaf=10,
    max_features="sqrt", n_jobs=-1, random_state=42
)
 
cv_cls_scores = cross_val_score(
    cv_cls, X_train_cls[:50000], y_cls_train[:50000],
    cv=5, scoring="accuracy", n_jobs=-1
)
 
print(f"   CV Accuracy per fold : {[round(s, 4) for s in cv_cls_scores]}")
print(f"   Mean CV Accuracy     : {cv_cls_scores.mean():.4f}")
print(f"   Std  CV Accuracy     : {cv_cls_scores.std():.4f}")
 
if cv_cls_scores.std() < 0.02:
    print(f"   → Stable across folds — no overfitting")
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP B6 — SAMPLE PREDICTIONS
# ─────────────────────────────────────────────────────────────────────────────
label_map_inv = {0: "Low", 1: "Medium", 2: "High"}
actual_cls  = [label_map_inv[v] for v in y_cls_test[idx]]
predict_cls = [label_map_inv[v] for v in y_test_pred_cls[idx]]
 
for i in range(10):
    match = "✅" if actual_cls[i] == predict_cls[i] else "❌"
    print(f"   {idx[i]:<5} {actual_vals[i]:>13.0f} {predict_vals[i]:>16.0f}"
          f" {actual_cls[i]:>12} {predict_cls[i]:>10} {match:>7}")

   Decision Tree            0.6348     0.6336     0.6362


D:\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


   Logistic Reg.            0.3432     0.3490     0.1725
   Random Forest            0.9174     0.8918     0.8911
   Training Random Forest Classifier...
Training complete

   Performance Metrics :

              precision    recall  f1-score   support

         Low       0.96      0.92      0.94     14520
      Medium       0.89      0.88      0.89     14520
        High       0.90      0.95      0.92     14960

    accuracy                           0.92     44000
   macro avg       0.92      0.92      0.92     44000
weighted avg       0.92      0.92      0.92     44000

   Actual Low         13,388         728         404
   Actual Medium         482      12,820       1,218
   Actual High            26         793      14,141

   Total correct : 40,349 / 44,000  (91.70%)
   CV Accuracy per fold : [np.float64(0.8739), np.float64(0.8588), np.float64(0.8757), np.float64(0.8541), np.float64(0.8469)]
   Mean CV Accuracy     : 0.8619
   Std  CV Accuracy     : 0.0112
   → Stable across fol

In [9]:
# ═════════════════════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ═════════════════════════════════════════════════════════════════════════════
print(f"""
   ┌───────────────────────────────────────────────────────────┐
   │             REGRESSION MODEL RESULTS                     │
   ├───────────────────────────┬───────────────────────────────┤
   │ Model                     │ Random Forest Regressor       │
   │ Train R²                  │ {r2_score(y_reg_train, y_train_pred_reg):.4f}                        │
   │ Test  R²                  │ {r2_score(y_reg_test, y_test_pred_reg):.4f}                        │
   │ Test  RMSE                │ {np.sqrt(mean_squared_error(y_reg_test, y_test_pred_reg)):.1f} kg/ha                   │
   │ Test  MAE                 │ {mean_absolute_error(y_reg_test, y_test_pred_reg):.1f} kg/ha                    │
   │ Overfit gap               │ {abs(r2_score(y_reg_train, y_train_pred_reg) - r2_score(y_reg_test, y_test_pred_reg)):.4f} (very low ✅)             │
   │ Cross-val R² (mean)       │ {cv_scores.mean():.4f} ± {cv_scores.std():.4f}              │
   └───────────────────────────┴───────────────────────────────┘
 
   ┌───────────────────────────────────────────────────────────┐
   │          CLASSIFICATION MODEL RESULTS                    │
   ├───────────────────────────┬───────────────────────────────┤
   │ Model                     │ Random Forest Classifier      │
   │ Train Accuracy            │ {tr_acc:.4f}                        │
   │ Test  Accuracy            │ {te_acc:.4f}                        │
   │ Test  F1-score (macro)    │ {te_f1:.4f}                        │
   │ Overfit gap               │ {abs(tr_acc - te_acc):.4f} (very low ✅)             │
   │ Cross-val Acc (mean)      │ {cv_cls_scores.mean():.4f} ± {cv_cls_scores.std():.4f}              │
   └───────────────────────────┴───────────────────────────────┘
 
   Why no overfitting?
   ✔ min_samples_leaf=10  → prevents trees fitting individual noise points
   ✔ max_features='sqrt'  → each tree sees different random feature subsets
   ✔ 100 trees averaged   → ensemble averaging cancels individual errors
   ✔ Train-test gap < 0.03 on both models → confirmed generalisation
   ✔ 5-fold CV shows consistent scores across all folds
""")


   ┌───────────────────────────────────────────────────────────┐
   │             REGRESSION MODEL RESULTS                     │
   ├───────────────────────────┬───────────────────────────────┤
   │ Model                     │ Random Forest Regressor       │
   │ Train R²                  │ 0.9719                        │
   │ Test  R²                  │ 0.9655                        │
   │ Test  RMSE                │ 1963.1 kg/ha                   │
   │ Test  MAE                 │ 788.0 kg/ha                    │
   │ Overfit gap               │ 0.0065 (very low ✅)             │
   │ Cross-val R² (mean)       │ 0.9578 ± 0.0023              │
   └───────────────────────────┴───────────────────────────────┘
 
   ┌───────────────────────────────────────────────────────────┐
   │          CLASSIFICATION MODEL RESULTS                    │
   ├───────────────────────────┬───────────────────────────────┤
   │ Model                     │ Random Forest Classifier      │
   │ Train Accuracy  